In [2]:
import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PositionalSharding
from functools import partial
import time

# --- Configuration (Optimized for Stability & Speed)
MAX_RECURSION_DEPTH = 1_000_000  # 🔥 Testing the highest recursion depth
OPTIMAL_DEPTH_STEP = 250_000  # 🔥 Breaking it into manageable steps
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE = 50_000_000  # 🔥 Extreme scaling with 50M samples per batch

@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    """Normalizes depth scaling to prevent instability."""
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    """🔥 Executes in optimized recursion chunks to maximize TPU efficiency"""
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))

    def body_fn(i, val):
        pi_dyn = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val

    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# --- TPU Sharding Setup ---
devices = jax.devices()
sharding = PositionalSharding(devices)

batch_input = jnp.linspace(0, 10, BATCH_SIZE)
batch_input = jax.device_put(batch_input, sharding)

batched_dppu_processing = pjit(
    lambda arr: vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=OPTIMAL_DEPTH_STEP, scale_factor=0.5), in_axes=0)(arr),
    in_shardings=(sharding,),
    out_shardings=sharding,
)

# ✅ **Optimized Adaptive Execution**
def process_with_larger_depths(x, total_depth):
    """🔥 Instead of running all at once, we now execute in 250K-depth steps"""
    iterations = total_depth // OPTIMAL_DEPTH_STEP
    for _ in range(iterations):
        x = dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP)
    return x

# --- 🚀 Optimized Execution ---
for depth in [250_000, 500_000, 1_000_000]:  # 🔥 Pushing TPU with massive depths
    start_time = time.time()
    output_batch = process_with_larger_depths(batch_input, depth)
    end_time = time.time()
    print(f"✅ Batch Output Shape (Depth={depth}):", output_batch.shape)
    print(f"🔥 Execution Time: {end_time - start_time:.6f} sec")

NUM_TRIALS = 2  # 🔥 Reduce trials to avoid unnecessary TPU overload

# Warm-up compile
_ = dppu_with_dynamic_pi_phi(jnp.ones((BATCH_SIZE,)), depth=OPTIMAL_DEPTH_STEP)

for depth in [250_000, 500_000, 1_000_000]:
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        result = process_with_larger_depths(jnp.ones((BATCH_SIZE,)), depth)
        _ = jax.device_get(result)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    print(f"\n🔥 TPU Benchmark (Depth={depth}, Batch={BATCH_SIZE})")
    print(f"Avg: {avg_time:.6f}, Min: {min(times):.6f}, Max: {max(times):.6f}")

# --- Investigate TPU Compilation Stability ---
compiled_fn_250k = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=250_000)
compiled_fn_1M = jax.jit(dppu_with_dynamic_pi_phi).lower(jnp.ones((BATCH_SIZE,)), depth=1_000_000)

print("\n🚀 XLA Compilation for Depth=250,000:")
print(compiled_fn_250k.as_text())

print("\n🚀 XLA Compilation for Depth=1,000,000:")
print(compiled_fn_1M.as_text())



✅ Batch Output Shape (Depth=250000): (50000000,)
🔥 Execution Time: 0.538627 sec
✅ Batch Output Shape (Depth=500000): (50000000,)
🔥 Execution Time: 0.000254 sec
✅ Batch Output Shape (Depth=1000000): (50000000,)
🔥 Execution Time: 0.000107 sec

🔥 TPU Benchmark (Depth=250000, Batch=50000000)
Avg: 119.205512, Min: 24.051906, Max: 214.359118

🔥 TPU Benchmark (Depth=500000, Batch=50000000)
Avg: 47.927411, Min: 47.921377, Max: 47.933444

🔥 TPU Benchmark (Depth=1000000, Batch=50000000)
Avg: 95.640604, Min: 95.638716, Max: 95.642492

🚀 XLA Compilation for Depth=250,000:
module @jit_dppu_with_dynamic_pi_phi attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<50000000xf32> {mhlo.layout_mode = "default"}, %arg1: tensor<i32> {mhlo.layout_mode = "default"}) -> (tensor<50000000xf32> {jax.result_info = "", mhlo.layout_mode = "default"}) {
    %0 = call @dppu_with_dynamic_pi_phi(%arg1, %arg0) : (tensor<i32>, tensor<50000000xf32>) -> tensor<500

/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/jax/_src/core.py:701: FutureWarning: unhashable type: <class 'jax._src.interpreters.partial_eval.DynamicJaxprTracer'>. Attempting to hash a tracer will lead to an error in a future JAX release.
  warnings.warn(
